# Stage D (Part 2) — Dynamic ERA5 **Daily** Features & Training Sets

### **Purpose**
Generate **daily basin-level meteorological features** from ERA5, join **static basin attributes**, and assemble **training datasets** (exogenous-only and ARX) for the modeled basins.

---

## **Inputs**
- **Dynamic daily features:**  
  `data/modeling/targets/train_basin_daily.parquet`  
  (basin × day table from Stage C, includes discharge and meteorology)

- **Static basin attributes:**  
  `data/modeling/static/basin_attributes.parquet`

---

## **Notebook Workflow**

1. **Setup**  
   Resolve project root, configure file paths, and set feature-engineering parameters (flux/state windows, API half-lives, ARX lags).

2. **Variable Catalog & Base Derivations**  
   - Classify variables as **flux-like** (precipitation, runoff, snowmelt, radiation, PET) or **state-like** (temperature, dewpoint, soil temperature, snow depth).  
   - Derive **wind_speed** (from U/V components) and **VPD** (from temperature & dewpoint).

3. **Leak-Safe Feature Engineering (Daily)**  
   - **Flux windows (sums):** 1/3/7/14/30-day past sums → `*_sum_{w}d`  
   - **State windows (means):** 1/3/7/14-day past means → `*_mean_{w}d`  
   - **Antecedent Precipitation Index (API):** exponential memory with 3- and 7-day half-lives → `api_d3`, `api_d7`  
   - **Calendar & condition flags:** `month`, `doy_sin`, `doy_cos`, `is_monsoon` (Jun–Sep), `is_freezing` (T ≤ 0 °C), `has_snowpack` (snow_depth > 0)

4. **Join Static Attributes**  
   Merge `basin_attributes.parquet` onto each `(basin_id, date_local)` row.

5. **Assemble Training Sets**  
   - **Exogenous-only:** features + target (`discharge_cms`, `qc_any`)  
   - **ARX (optional):** add discharge lags `q_lag_{1,2,3,7,14,30}d` and rolling stats (`q_roll7_mean`, `q_roll14_std`), dropping rows without full lag history.

6. **Training datasets:**  
   - Exogenous → `data/modeling/datasets/train_basin_daily_exogenous.parquet`  
   - ARX (if enabled) → `data/modeling/datasets/train_basin_daily_arx.parquet`

---

## **Key Tunable Parameters**
- **Flux windows:** `[1, 3, 7, 14, 30]`
- **State windows:** `[1, 3, 7, 14]`
- **API half-lives:** `[3, 7]`
- **ARX lags:** `[1, 2, 3, 7, 14, 30]`
- **ARX rolling stats:** `(7, mean)`, `(14, std)`

In [1]:
from pathlib import Path
import subprocess
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# === Resolve Project Root ===

def get_project_root(max_up=6):
    try:
        root = subprocess.check_output(["git","rev-parse","--show-toplevel"], text=True).strip()
        if root:
            return Path(root)
    except Exception:
        pass
    p = Path.cwd()
    for _ in range(max_up):
        if (p/"data").exists() and (p/"code").exists():
            return p
        if (p/".git").exists():
            return p
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = get_project_root()
print("Project root:", PROJECT_ROOT)


Project root: /Users/qingfangliu/bhutan_climate_modeling


In [3]:
# === Paths & Settings ===

# Inputs
FEATURES_DAILY_PARQUET = PROJECT_ROOT / "data/modeling/targets/train_basin_daily.parquet"
STATIC_ATTR_PARQUET   = PROJECT_ROOT / "data/modeling/static/basin_attributes.parquet"

In [4]:
# Outputs

TRAIN_EXOG_PARQUET     = PROJECT_ROOT / "data/modeling/datasets/train_basin_daily_exogenous.parquet"
TRAIN_ARX_PARQUET      = PROJECT_ROOT / "data/modeling/datasets/train_basin_daily_arx.parquet"

TRAIN_EXOG_PARQUET.parent.mkdir(parents=True, exist_ok=True)

In [5]:
# Feature knobs (daily cadence)
FLUX_WINDOWS  = [1, 3, 7, 14, 30]      # sums of past N full days
STATE_WINDOWS = [1, 3, 7, 14]          # means of past N full days
API_HALFLIFE  = [3, 7]                 # days (for antecedent precip index)
ARX_LAGS      = [1, 2, 3, 7, 14, 30]   # discharge lags (days)
ROLL_Q_STATS  = [(7, "mean"), (14, "std")]  # discharge rolling stats
PRODUCE_ARX   = True                   # also build ARX dataset (exogenous + lagged Q)

## Variable Catalog & Base Derivations (wind, VPD)

This block classifies the numeric ERA5 variables into **flux-like** (e.g., precipitation, runoff, radiation) and **state-like** (e.g., temperature, soil moisture) features for modeling.  
It also derives additional useful features:  
* **wind_speed** from available U/V wind components  
* **vpd_kpa** (vapor pressure deficit) from temperature and dewpoint, auto-detecting Kelvin vs °C  

Finally, it prints sample lists of flux and state variables, helping verify which features will be used downstream.


In [6]:
# Load features + targets
features = pd.read_parquet(FEATURES_DAILY_PARQUET)

# --- Pick numeric candidate columns (exclude IDs / time / unwanted vars) ---
id_like = {"basin_id", "basin_name", "date_local"}
exclude_cols = {"discharge_cms", "qc_any", "n_cells", "low_coverage"}

num_cols = [
    c for c in features.columns
    if c not in id_like
    and c not in exclude_cols
    and pd.api.types.is_numeric_dtype(features[c])
]

num_cols

['temperature',
 'dewpoint',
 'wind_u',
 'wind_v',
 'potential_evaporation',
 'runoff',
 'snow_depth',
 'snowmelt',
 'soil_temperature',
 'sub_surface_runoff',
 'surface_runoff',
 'solar_radiation',
 'precipitation']

In [7]:
# --- Heuristic classification: flux vs state ---
def classify_flux_state(cols):
    flux_like, state_like = set(), set()
    for c in cols:
        lc = c.lower()
        if any(k in lc for k in ["precip", "runoff", "snowmelt", "evap", "radiation", "ssrd"]):
            flux_like.add(c)
        else:
            state_like.add(c)
    return sorted(flux_like), sorted(state_like)

flux_vars, state_vars = classify_flux_state(num_cols)

print(f"Flux-like variables ({len(flux_vars)}):")
for v in flux_vars:
    print(f"  • {v}")

print(f"\nState-like variables ({len(state_vars)}):")
for v in state_vars:
    print(f"  • {v}")

Flux-like variables (7):
  • potential_evaporation
  • precipitation
  • runoff
  • snowmelt
  • solar_radiation
  • sub_surface_runoff
  • surface_runoff

State-like variables (6):
  • dewpoint
  • snow_depth
  • soil_temperature
  • temperature
  • wind_u
  • wind_v


In [8]:
# --- Derive wind_speed if U/V present (kept as state-like) ---
def add_wind_speed(df, state_vars):
    candidates = [
        ("u10", "v10"),
        ("u_component_of_wind_10m", "v_component_of_wind_10m"),
        ("wind_u", "wind_v"), ("u", "v"),
    ]
    for u_name, v_name in candidates:
        u = next((c for c in df.columns if u_name == c or u_name in c.lower()), None)
        v = next((c for c in df.columns if v_name == c or v_name in c.lower()), None)
        if u and v:
            if "wind_speed" not in df.columns:
                df["wind_speed"] = (df[u]**2 + df[v]**2) ** 0.5
            if "wind_speed" not in state_vars:
                state_vars = state_vars + ["wind_speed"]
            break
    if "wind_speed" in df.columns and "wind_speed" not in state_vars:
        state_vars = state_vars + ["wind_speed"]
    return df, state_vars

features, state_vars = add_wind_speed(features, state_vars)

In [9]:
# --- Derive VPD (auto-detect Kelvin vs °C) ---
def to_celsius(series: pd.Series) -> pd.Series:
    med = series.median(skipna=True)
    return series - 273.15 if pd.notna(med) and med > 200 else series

def find_col(df, name_hints):
    for h in name_hints:
        c = next((c for c in df.columns if h == c or h in c.lower()), None)
        if c: return c
    return None

temp_col = find_col(features, ["t2m","temperature","2m_temperature"])
dew_col  = find_col(features, ["d2m","dewpoint","dew_point","2m_dewpoint"])

if temp_col and dew_col and "vpd_kpa" not in features.columns:
    T_c  = to_celsius(features[temp_col])
    Td_c = to_celsius(features[dew_col])
    es = 0.6108 * np.exp(17.27 * T_c  / (T_c  + 237.3))
    ea = 0.6108 * np.exp(17.27 * Td_c / (Td_c + 237.3))
    features["vpd_kpa"] = (es - ea).clip(lower=0)
    if "vpd_kpa" not in state_vars:
        state_vars.append("vpd_kpa")

print("Flux-like vars:", flux_vars)
print("State-like vars (incl. derived):", [v for v in state_vars if v not in flux_vars])

Flux-like vars: ['potential_evaporation', 'precipitation', 'runoff', 'snowmelt', 'solar_radiation', 'sub_surface_runoff', 'surface_runoff']
State-like vars (incl. derived): ['dewpoint', 'snow_depth', 'soil_temperature', 'temperature', 'wind_u', 'wind_v', 'wind_speed', 'vpd_kpa']


## Rolling & Lag Functions (Leak-safe)

This block defines **leak-safe rolling and lag functions** for time-series features, ensuring that only information from past days is used (no look-ahead bias).  

* `strict_sum_past` – rolling sum over the previous *N* full days, excluding today.  
* `strict_mean_past` – rolling mean over the previous *N* full days, excluding today.  
* `api_series` – computes an **Antecedent Precipitation Index (API)** or similar exponentially decaying memory of past fluxes, shifted by one day so that only data up to *t−1* is used.


In [10]:

# === Rolling & lag helpers (add once, then use as needed) ===
def strict_sum_past(s: pd.Series, window: int) -> pd.Series:
    """Rolling sum over the previous N full days, excluding today."""
    return s.shift(1).rolling(window=window, min_periods=window).sum()

def strict_mean_past(s: pd.Series, window: int) -> pd.Series:
    """Rolling mean over the previous N full days, excluding today."""
    return s.shift(1).rolling(window=window, min_periods=window).mean()

def api_series(s: pd.Series, half_life_days: int) -> pd.Series:
    """Antecedent Precipitation Index (API) using exponential decay, shifted to avoid look-ahead leakage."""
    k = 0.5 ** (1.0 / half_life_days)
    out = np.empty(len(s))
    out[:] = np.nan
    acc = 0.0
    for i, val in enumerate(s.fillna(0.0).values):
        acc = val + k * acc
        out[i] = acc
    return pd.Series(out, index=s.index).shift(1)


In [11]:
features.columns

Index(['basin_id', 'basin_name', 'date_local', 'discharge_cms', 'qc_any',
       'temperature', 'dewpoint', 'wind_u', 'wind_v', 'potential_evaporation',
       'runoff', 'snow_depth', 'snowmelt', 'soil_temperature',
       'sub_surface_runoff', 'surface_runoff', 'solar_radiation',
       'precipitation', 'n_cells', 'low_coverage', 'wind_speed', 'vpd_kpa'],
      dtype='object')

In [12]:
# === Build engineered features per basin_id ===
e = features.sort_values(["basin_id","date_local"]).reset_index(drop=True)

# Work on a copy we will extend
feat = e[["basin_id","date_local"]].copy()

# Flux windows (past-only rolling sums)
for v in flux_vars:
    if v in e.columns:
        for w in FLUX_WINDOWS:
            col = f"{v}_sum_{w}d"
            feat[col] = (e.groupby("basin_id")[v]
                           .apply(lambda s: strict_sum_past(s, w))
                           .reset_index(level=0, drop=True))

# State windows (past-only rolling means)
for v in state_vars:
    if v in e.columns:
        for w in STATE_WINDOWS:
            col = f"{v}_mean_{w}d"
            feat[col] = (e.groupby("basin_id")[v]
                           .apply(lambda s: strict_mean_past(s, w))
                           .reset_index(level=0, drop=True))

# API from precip (if present)
precip_col = next((c for c in e.columns if "precip" in c.lower()), None)
if precip_col:
    for h in API_HALFLIFE:
        col = f"api_d{h}"
        feat[col] = (e.groupby("basin_id")[precip_col]
                       .apply(lambda s: api_series(s, h))
                       .reset_index(level=0, drop=True))
        
        

In [13]:
feat.columns

Index(['basin_id', 'date_local', 'potential_evaporation_sum_1d',
       'potential_evaporation_sum_3d', 'potential_evaporation_sum_7d',
       'potential_evaporation_sum_14d', 'potential_evaporation_sum_30d',
       'precipitation_sum_1d', 'precipitation_sum_3d', 'precipitation_sum_7d',
       'precipitation_sum_14d', 'precipitation_sum_30d', 'runoff_sum_1d',
       'runoff_sum_3d', 'runoff_sum_7d', 'runoff_sum_14d', 'runoff_sum_30d',
       'snowmelt_sum_1d', 'snowmelt_sum_3d', 'snowmelt_sum_7d',
       'snowmelt_sum_14d', 'snowmelt_sum_30d', 'solar_radiation_sum_1d',
       'solar_radiation_sum_3d', 'solar_radiation_sum_7d',
       'solar_radiation_sum_14d', 'solar_radiation_sum_30d',
       'sub_surface_runoff_sum_1d', 'sub_surface_runoff_sum_3d',
       'sub_surface_runoff_sum_7d', 'sub_surface_runoff_sum_14d',
       'sub_surface_runoff_sum_30d', 'surface_runoff_sum_1d',
       'surface_runoff_sum_3d', 'surface_runoff_sum_7d',
       'surface_runoff_sum_14d', 'surface_runoff_sum_3

In [14]:
# Calendar / seasonality
feat["month"]   = feat["date_local"].dt.month
doy             = feat["date_local"].dt.dayofyear
feat["doy_sin"] = np.sin(2*np.pi * (doy/365.25))
feat["doy_cos"] = np.cos(2*np.pi * (doy/365.25))
feat["is_monsoon"] = feat["month"].between(6,9).astype(int)

# Condition flags (freezing / snowpack) from original ERA5 columns
snow_depth_col = next((c for c in e.columns if "snow_depth" in c.lower()), None)
temp_col       = next((c for c in e.columns if c.lower().startswith(("t2m","temperature","2m_temperature"))), None)

if temp_col:
    T_c = to_celsius(e[temp_col])
    e["is_freezing_flag"] = (T_c <= 0).astype(int)
    feat["is_freezing"] = e.groupby("basin_id")["is_freezing_flag"].shift(1)

if snow_depth_col:
    e["has_snowpack_flag"] = (e[snow_depth_col] > 0).astype(int)
    feat["has_snowpack"] = e.groupby("basin_id")["has_snowpack_flag"].shift(1)

feat.columns

Index(['basin_id', 'date_local', 'potential_evaporation_sum_1d',
       'potential_evaporation_sum_3d', 'potential_evaporation_sum_7d',
       'potential_evaporation_sum_14d', 'potential_evaporation_sum_30d',
       'precipitation_sum_1d', 'precipitation_sum_3d', 'precipitation_sum_7d',
       'precipitation_sum_14d', 'precipitation_sum_30d', 'runoff_sum_1d',
       'runoff_sum_3d', 'runoff_sum_7d', 'runoff_sum_14d', 'runoff_sum_30d',
       'snowmelt_sum_1d', 'snowmelt_sum_3d', 'snowmelt_sum_7d',
       'snowmelt_sum_14d', 'snowmelt_sum_30d', 'solar_radiation_sum_1d',
       'solar_radiation_sum_3d', 'solar_radiation_sum_7d',
       'solar_radiation_sum_14d', 'solar_radiation_sum_30d',
       'sub_surface_runoff_sum_1d', 'sub_surface_runoff_sum_3d',
       'sub_surface_runoff_sum_7d', 'sub_surface_runoff_sum_14d',
       'sub_surface_runoff_sum_30d', 'surface_runoff_sum_1d',
       'surface_runoff_sum_3d', 'surface_runoff_sum_7d',
       'surface_runoff_sum_14d', 'surface_runoff_sum_3

In [15]:
feat.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29840 entries, 0 to 29839
Data columns (total 77 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   basin_id                       29840 non-null  float64       
 1   date_local                     29840 non-null  datetime64[ns]
 2   potential_evaporation_sum_1d   29837 non-null  float64       
 3   potential_evaporation_sum_3d   29831 non-null  float64       
 4   potential_evaporation_sum_7d   29819 non-null  float64       
 5   potential_evaporation_sum_14d  29798 non-null  float64       
 6   potential_evaporation_sum_30d  29750 non-null  float64       
 7   precipitation_sum_1d           29837 non-null  float64       
 8   precipitation_sum_3d           29831 non-null  float64       
 9   precipitation_sum_7d           29819 non-null  float64       
 10  precipitation_sum_14d          29798 non-null  float64       
 11  precipitation_s

## Join Static Basin Attributes

In [16]:
# Join static attributes
static = pd.read_parquet(STATIC_ATTR_PARQUET)

# Join on both basin_id and basin_name
final = feat.merge(
    static,
    on="basin_id",
    how="left",
    validate="m:1"  # ensures many-to-one merge: many daily rows per one basin
)

print(final.columns.tolist())

['basin_id', 'date_local', 'potential_evaporation_sum_1d', 'potential_evaporation_sum_3d', 'potential_evaporation_sum_7d', 'potential_evaporation_sum_14d', 'potential_evaporation_sum_30d', 'precipitation_sum_1d', 'precipitation_sum_3d', 'precipitation_sum_7d', 'precipitation_sum_14d', 'precipitation_sum_30d', 'runoff_sum_1d', 'runoff_sum_3d', 'runoff_sum_7d', 'runoff_sum_14d', 'runoff_sum_30d', 'snowmelt_sum_1d', 'snowmelt_sum_3d', 'snowmelt_sum_7d', 'snowmelt_sum_14d', 'snowmelt_sum_30d', 'solar_radiation_sum_1d', 'solar_radiation_sum_3d', 'solar_radiation_sum_7d', 'solar_radiation_sum_14d', 'solar_radiation_sum_30d', 'sub_surface_runoff_sum_1d', 'sub_surface_runoff_sum_3d', 'sub_surface_runoff_sum_7d', 'sub_surface_runoff_sum_14d', 'sub_surface_runoff_sum_30d', 'surface_runoff_sum_1d', 'surface_runoff_sum_3d', 'surface_runoff_sum_7d', 'surface_runoff_sum_14d', 'surface_runoff_sum_30d', 'dewpoint_mean_1d', 'dewpoint_mean_3d', 'dewpoint_mean_7d', 'dewpoint_mean_14d', 'snow_depth_mean_1

In [17]:
# Build a compact summary: column, dtype, % missing
summary = pd.DataFrame({
    "dtype": final.dtypes,
    "missing_count": final.isna().sum(),
    "missing_pct": final.isna().mean() * 100
}).reset_index().rename(columns={"index": "column"})

# Sort by % missing, descending
summary_sorted = summary.sort_values(by="missing_pct", ascending=False)

# Display nicely
pd.set_option("display.max_rows", None)  # show all rows
display(summary_sorted)

,column,dtype,missing_count,missing_pct
21,snowmelt_sum_30d,float64,90,0.301609
36,surface_runoff_sum_30d,float64,90,0.301609
31,sub_surface_runoff_sum_30d,float64,90,0.301609
6,potential_evaporation_sum_30d,float64,90,0.301609
26,solar_radiation_sum_30d,float64,90,0.301609
11,precipitation_sum_30d,float64,90,0.301609
16,runoff_sum_30d,float64,90,0.301609
15,runoff_sum_14d,float64,42,0.140751
40,dewpoint_mean_14d,float64,42,0.140751
35,surface_runoff_sum_14d,float64,42,0.140751


## Prepare to Build Train

In [18]:
# ----------------------------------------
# 0) Pick the engineered feature table
#    Use `final` if you merged statics; fallback to `feat`
# ----------------------------------------
feat_table = final if 'final' in globals() else feat

# Safety: ensure keys exist
assert {'basin_id','date_local'}.issubset(feat_table.columns), "feat/final missing keys"

# ----------------------------------------
# 1) Build y directly from `features`
# ----------------------------------------
target_cols = ['discharge_cms']
if 'qc_any' in features.columns:
    target_cols.append('qc_any')

y = (features[['basin_id','date_local', *target_cols]]
     .drop_duplicates()
     .sort_values(['basin_id','date_local'])
     .reset_index(drop=True))

## Build Train (Exogenous-Only)

In [19]:
feat_sorted = (feat_table
               .sort_values(['basin_id','date_local'])
               .reset_index(drop=True))

train_exog = (feat_sorted
              .merge(y, on=['basin_id','date_local'], how='left', validate='m:1')
              .sort_values(['basin_id','date_local'])
              .reset_index(drop=True))

# Optional: filter out rows where qc flags are True
if 'qc_any' in train_exog.columns:
    train_exog = train_exog.loc[~train_exog['qc_any'].fillna(False)].reset_index(drop=True)

print(f"[EXOG] rows: {len(train_exog)} | missing target: {train_exog['discharge_cms'].isna().sum()}")

train_exog.to_parquet(TRAIN_EXOG_PARQUET, index=False)
print("Wrote exogenous train →", TRAIN_EXOG_PARQUET)

[EXOG] rows: 29185 | missing target: 42
Wrote exogenous train → /Users/qingfangliu/bhutan_climate_modeling/data/modeling/datasets/train_basin_daily_exogenous.parquet


## Build ARX (add lagged discharge), leak-safe

This step augments the exogenous feature set with **auto-regressive (AR) terms**:
- **Lagged discharge (`q_lag_Xd`)** – yesterday’s flow, last week’s flow, etc. – to capture flow persistence.  
- **Rolling statistics (`q_rollX_mean`, `q_rollX_std`)** – mean and variability of recent flow, computed on shifted (past-only) discharge to avoid leakage.

Including these features helps the model learn temporal dependencies and improves prediction skill, especially where streamflow is highly autocorrelated.


In [20]:
if PRODUCE_ARX:
    te = train_exog.copy()
    te = te.sort_values(['basin_id','date_local']).reset_index(drop=True)

    g = te.groupby('basin_id', group_keys=False)

    # Add discharge lags
    for L in ARX_LAGS:  # e.g., [1,2,3,7,14,30]
        te[f"q_lag_{L}d"] = g['discharge_cms'].shift(L)

    # Rolling stats on *shifted* discharge (past-only)
    for win, stat in ROLL_Q_STATS:  # e.g., [(7,"mean"), (14,"std")]
        shifted = g['discharge_cms'].shift(1)
        if stat == "mean":
            te[f"q_roll{win}_mean"] = shifted.rolling(win, min_periods=win).mean()
        elif stat == "std":
            te[f"q_roll{win}_std"]  = shifted.rolling(win, min_periods=win).std()

    # Drop rows without full lag history or missing target
    needed = [f"q_lag_{L}d" for L in ARX_LAGS]
    before = len(te)
    te = te.dropna(subset=['discharge_cms'] + needed).reset_index(drop=True)
    after = len(te)
    print(f"[ARX] rows: {after} (dropped {before-after} for lag history / missing target)")

    # Save if you have paths defined
    if 'TRAIN_ARX_PARQUET' in globals():
        te.to_parquet(TRAIN_ARX_PARQUET, index=False)
        print("Wrote ARX train →", TRAIN_ARX_PARQUET)
else:
    print("PRODUCE_ARX=False → skipping ARX dataset.")    

[ARX] rows: 29011 (dropped 174 for lag history / missing target)
Wrote ARX train → /Users/qingfangliu/bhutan_climate_modeling/data/modeling/datasets/train_basin_daily_arx.parquet
